# DCC Mission 2 — 신고자 / 119대원 발화 분류

한 사람의 한 발화 조각(startAt~endAt)을 듣고 신고자(speaker=1) / 119대원(speaker=0)을 분류합니다. 평가 지표는 Accuracy입니다.

```text
대회 제공 zip ─ prepare_data.py: 통화의 wav·json 을 꺼냄 (Training 은 서울만, Validation 은 평가 전용)
   ▼
발화 구간 (startAt~endAt, 16kHz, 최대 6초) ─ 한 발화 = 한 입력
   ▼
WavLM-base-plus 앞 4층 ─ 층별 평균·표준편차 ─ 층 가중합 ─ 분류기 ─ 확률 ≥ 0.5 → 신고자(1)
   학습: ① WavLM 동결, 분류기만 학습 → ② Transformer 위쪽 2층 미세조정 → 내부 dev 로 선택
```

| 규칙 | 구현 |
|---|---|
| 입력은 startAt/endAt로 자른 한 발화 | 통화 wav에서 그 구간만 읽어 발화마다 독립적으로 예측 |
| startAt/endAt 외 annotation·발화 순서 사용 금지 | json에서 startAt/endAt(학습 때만 정답 speaker)만 읽음. wav는 같은 파일 이름으로만 연결 |
| Training은 서울 데이터만, Validation 학습 금지 | Training은 경로에 `서울`이 있는 파일만. Validation은 `inference.py` 평가에만 사용 |
| label에 따른 전처리 금지 | 통화·발화 선택과 전처리가 label을 보지 않음. 학습과 추론의 입력 처리가 같음 |
| Decision Threshold 0.5 고정 | 확률 ≥ 0.5 → 1. threshold 탐색 없음 |
| `inference.py` 1회 실행, 인터넷 불필요 | 체크포인트에 WavLM 설정·가중치 포함. 마지막 셀에서 인터넷을 막고 확인 |
| 모델 효율 기록 | 파라미터 수, 실행 환경, batch size, Validation 추론 시간을 `efficiency_report.md`에 저장 |

위에서 아래로 실행합니다.

In [ ]:
# 코드 파일을 저장할 폴더
import json, os, platform, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

PROJECT = Path('/content/mission2')
PROJECT.mkdir(parents=True, exist_ok=True)
print('코드 저장 위치:', PROJECT)

코드 저장 위치: /content/mission2


## 1. 코드 파일 저장

아래 `%%writefile` 셀들이 제출용 코드 파일을 만듭니다.

### requirements.txt — 필요 패키지

In [ ]:
%%writefile "{PROJECT}/requirements.txt"
# Python 3.10+. torch 는 실행 환경(CPU/CUDA)에 맞는 2.6 이상 버전을 사용합니다.
torch>=2.6,<3
transformers==4.57.1
soundfile==0.13.1
numpy>=1.26,<3
scipy>=1.11,<2

Writing /content/mission2/requirements.txt


### model.py — 발화 목록, 발화 구간 음성 처리, 모델, 예측, 체크포인트

In [ ]:
%%writefile "{PROJECT}/model.py"
"""
model.py — Mission 2 (신고자 / 119대원 발화 분류) 모델과 공용 함수

입력 : 통화 wav 에서 라벨링데이터의 startAt~endAt(ms) 구간만 잘라낸 발화 조각 1개
출력 : 1 = 신고자, 0 = 119대원  (확률 >= 0.5 → 1, threshold 고정)
라벨 json 에서 읽는 값 : startAt, endAt (학습 때만 정답 speaker). text·발화 순서 등 다른 annotation 은 읽지 않음

모델
  발화 구간(16kHz, 최대 6초) → WavLM-base-plus 의 CNN + Transformer 앞 4층
    → 층마다(임베딩 + 4층 = 5개) 패딩을 뺀 평균·표준편차(1536차원)
    → 층별 표준화 → softmax 층 가중합 → LayerNorm → Linear 256 → GELU → Dropout → Linear 1 → sigmoid
  아래쪽 층만 쓰는 이유: 신고자(휴대폰·현장)와 119대원(상황실 헤드셋)을 가르는 채널·음색 정보가 아래 층에 남아 있다.

체크포인트 한 파일에 WavLM 설정·가중치와 분류기가 모두 들어 있어 추론 때 인터넷이 필요 없다.
"""
import glob
import json
import math
import os
import time
import warnings
from collections import defaultdict

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2FeatureExtractor, WavLMConfig, WavLMModel

warnings.filterwarnings("ignore", message="Support for mismatched key_padding_mask and attn_mask")

SR = 16000                      # WavLM 입력 샘플레이트 (원본 8kHz 를 변환)
WINDOW = 6 * SR                 # 모델 입력 최대 길이 6초 (더 긴 발화는 가운데 6초)
BUCKET = SR // 4                # 0.25초 단위로 zero padding → 같은 길이끼리 batch
MIN_SAMPLES = 400               # WavLM CNN 이 프레임 1개를 만드는 최소 길이
THRESHOLD = 0.5                 # 확률 >= 0.5 → 신고자(1). 조정하지 않는 고정값
PRETRAINED = "microsoft/wavlm-base-plus"
REVISION = "4c66d4806a428f2e922ccfa1a962776e232d487b"
FORMAT = "mission2-wavlm-layermix-v1"


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================
# 1. 발화 목록 (라벨 json ↔ 같은 이름의 wav)
# ============================================================
def _sample_index(value, sr):
    """startAt/endAt(ms) → 샘플 위치. 숫자가 아니면 None."""
    try:
        return int(float(value) * sr / 1000)
    except (TypeError, ValueError):
        return None


def build_manifest(label_dir, audio_dir, with_label):
    """라벨 json 의 utterances 를 발화 1개 = 1행으로 펼친다.

    - wav 매칭: json 과 같은 파일 이름 (다른 annotation 으로 찾지 않음)
    - 구간이 wav 길이를 넘으면 wav 끝에 맞춰 자른다
    - with_label=True (학습): 정답이 0/1 이 아니거나 구간을 읽을 수 없는 발화는 제외
    - with_label=False (추론): 어떤 발화도 빼지 않는다. 읽을 수 없는 발화는 ok=False 로 표시 → 입력 발화마다 CSV 1행
    행: call, wav, wav_name, start, end(입력 값 그대로), a, b(wav 안의 샘플 구간), sr, ok, [speaker]
    """
    wavs = {}
    for path in sorted(glob.glob(os.path.join(audio_dir, "**", "*"), recursive=True)):
        if path.lower().endswith(".wav"):
            wavs.setdefault(os.path.splitext(os.path.basename(path))[0], path)
    rows, dropped, clipped, unreadable = [], 0, 0, 0
    for label_path in sorted(glob.glob(os.path.join(label_dir, "**", "*"), recursive=True)):
        if not label_path.lower().endswith(".json"):
            continue
        call = os.path.splitext(os.path.basename(label_path))[0]
        try:
            with open(label_path, encoding="utf-8-sig") as f:
                utterances = json.load(f)["utterances"]
            assert isinstance(utterances, list)
        except Exception:
            unreadable += 1                      # 발화 목록을 읽을 수 없는 json
            continue
        wav, frames, sr = wavs.get(call), 0, SR
        if wav is not None:
            try:
                info = sf.info(wav)
                frames, sr = info.frames, info.samplerate
            except Exception:
                wav = None                       # 열 수 없는 wav
        for u in utterances:
            u = u if isinstance(u, dict) else {}
            a, b = _sample_index(u.get("startAt"), sr), _sample_index(u.get("endAt"), sr)
            ok = wav is not None and a is not None and b is not None
            if ok:
                clipped += b > frames
                a, b = max(a, 0), min(b, frames)
                ok = b > a
            row = dict(call=call, wav=wav, wav_name=os.path.basename(wav) if wav else call + ".wav",
                       start=u.get("startAt"), end=u.get("endAt"), a=a if ok else 0, b=b if ok else 0, sr=sr, ok=ok)
            if with_label:
                if not ok or str(u.get("speaker")) not in ("0", "1"):
                    dropped += 1
                    continue
                row["speaker"] = int(u["speaker"])
            rows.append(row)
    log(f"발화 목록: 통화 {len({r['call'] for r in rows}):,}개 / 발화 {len(rows):,}개"
        + (f" / 제외 {dropped:,}개" if dropped else "") + (f" / wav 끝에 맞춰 자른 발화 {clipped:,}개" if clipped else "")
        + (f" / 읽을 수 없는 json {unreadable:,}개" if unreadable else ""))
    return rows


# ============================================================
# 2. 발화 구간 음성 → 모델 입력
# ============================================================
def resampled_length(row):
    """발화 구간을 16kHz 로 바꿨을 때의 샘플 수."""
    g = math.gcd(row["sr"], SR)
    return -(-(row["b"] - row["a"]) * (SR // g) // (row["sr"] // g))


def padded_length(n):
    """padding 후 길이: 0.25초 배수로 올림, 최대 6초.

    WavLM-base 의 CNN 은 입력 전체 길이로 정규화하므로 padding 양이 결과에 영향을 준다.
    padding 후 길이를 발화 자신의 길이로만 정하면, 어떤 발화와 같은 batch 에 들어가든 결과가 같다.
    """
    return min(WINDOW, -(-max(n, MIN_SAMPLES) // BUCKET) * BUCKET)


def load_segment(row):
    """wav 에서 발화 구간만 읽어 mono·16kHz 로 변환 (통화의 다른 부분은 읽지 않음)."""
    with sf.SoundFile(row["wav"]) as f:
        f.seek(row["a"])
        x = f.read(row["b"] - row["a"], dtype="float32", always_2d=True).mean(axis=1)
    if row["sr"] != SR:
        g = math.gcd(row["sr"], SR)
        x = resample_poly(x, SR // g, row["sr"] // g).astype(np.float32)
    return np.nan_to_num(x)


class Segments(Dataset):
    """발화 1개 → (위치, zero padding 된 음성, 실제 길이). label 은 읽지 않는다.

    6초보다 긴 발화: 추론은 가운데 6초, 학습(crop_seed 지정)은 무작위 6초.
    """
    def __init__(self, rows, do_normalize, crop_seed=None):
        self.rows, self.do_normalize, self.crop_seed = rows, do_normalize, crop_seed

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        x = load_segment(self.rows[i])
        if len(x) > WINDOW:
            start = (len(x) - WINDOW) // 2 if self.crop_seed is None else \
                int(np.random.default_rng([self.crop_seed, i]).integers(0, len(x) - WINDOW + 1))
            x = x[start:start + WINDOW]
        if self.do_normalize:
            x = (x - x.mean()) / np.sqrt(x.var() + 1e-7)
        padded = np.zeros(padded_length(len(x)), dtype=np.float32)
        padded[:len(x)] = x
        return i, padded, max(len(x), MIN_SAMPLES)


def length_batches(rows, batch_size, rng=None):
    """padding 후 길이가 같은 발화끼리 batch 로 묶는다. rng 를 주면(학습) 순서를 섞는다."""
    groups = defaultdict(list)
    for i, row in enumerate(rows):
        groups[padded_length(resampled_length(row))].append(i)
    batches = []
    for length in sorted(groups):
        order = groups[length] if rng is None else rng.permutation(groups[length]).tolist()
        batches += [order[k:k + batch_size] for k in range(0, len(order), batch_size)]
    return batches if rng is None else [batches[i] for i in rng.permutation(len(batches))]


def collate(items):
    width = max(len(item[1]) for item in items)
    waveform = np.zeros((len(items), width), dtype=np.float32)
    for k, item in enumerate(items):
        waveform[k, :len(item[1])] = item[1]
    return [item[0] for item in items], torch.from_numpy(waveform), torch.tensor([item[2] for item in items])


# ============================================================
# 3. 모델
# ============================================================
class RoleModel(nn.Module):
    """WavLM 앞 K층의 층별 통계 + 층 가중합 분류기. forward 출력은 logit (sigmoid 전)."""

    def __init__(self, config, do_normalize=False, use_attention_mask=True, head_dim=256, dropout=0.2):
        super().__init__()
        if config.add_adapter or config.do_stable_layer_norm:
            raise ValueError("WavLM-base 계열 구조만 지원합니다")
        config.apply_spec_augment = False                  # 무작위 마스킹·layerdrop 끔: 같은 입력 → 같은 출력
        config.mask_time_prob = config.mask_feature_prob = config.layerdrop = 0.0
        self.wavlm = WavLMModel(config)
        self.do_normalize, self.use_attention_mask = bool(do_normalize), bool(use_attention_mask)
        layers, dim = config.num_hidden_layers + 1, 2 * config.hidden_size
        # 층별 표준화 통계: 학습 데이터에서 한 번 계산해 고정 (학습 파라미터 아님)
        self.register_buffer("stat_mean", torch.zeros(layers, dim))
        self.register_buffer("stat_scale", torch.ones(layers, dim))
        self.layer_logits = nn.Parameter(torch.zeros(layers))             # softmax → 층 가중치
        self.norm = nn.LayerNorm(dim)
        self.classifier = nn.Sequential(nn.Linear(dim, head_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(head_dim, 1))
        self.finetuned = False
        self.set_finetune(False)

    @classmethod
    def from_pretrained(cls, source=PRETRAINED, layers=4):
        """사전학습 WavLM 에서 앞 layers 층만 가져온다 (학습할 때만 사용, 인터넷 또는 로컬 폴더)."""
        pin = {} if os.path.isdir(source) else {"revision": REVISION}
        full = WavLMModel.from_pretrained(source, **pin)
        processor = Wav2Vec2FeatureExtractor.from_pretrained(source, **pin)
        config = WavLMConfig.from_dict(full.config.to_dict())
        config.num_hidden_layers = layers
        model = cls(config, processor.do_normalize, processor.return_attention_mask)
        # 앞 층의 출력은 뒤 층과 무관하므로, 뒤 층을 버려도 남은 층의 출력은 그대로다
        keep = {k: v for k, v in full.state_dict().items() if k in model.wavlm.state_dict()}
        model.wavlm.load_state_dict(keep, strict=True)
        return model

    def head_parameters(self):
        return [self.layer_logits, *self.norm.parameters(), *self.classifier.parameters()]

    def tuned_modules(self):
        """미세조정 대상: Transformer 위쪽 2층. 그 아래(CNN 특징 추출기·projection·아래 층)는 항상 동결."""
        return list(self.wavlm.encoder.layers[-2:])

    def set_finetune(self, enabled):
        self.finetune = bool(enabled)
        self.wavlm.requires_grad_(False)
        if self.finetune:
            self.finetuned = True
            for module in self.tuned_modules():
                module.requires_grad_(True)
        return self.train(self.training)

    def train(self, mode=True):
        """동결된 부분은 항상 eval(dropout 끔). 미세조정 중일 때만 미세조정 대상 층이 train 모드."""
        super().train(mode)
        self.wavlm.eval()
        if mode and self.finetune:
            for module in self.tuned_modules():
                module.train()
        return self

    def layer_stats(self, waveform, lengths):
        """[batch, 샘플] → 층별 평균·표준편차 [batch, 층 수, 2·hidden]. 동결이면 gradient 를 만들지 않는다."""
        with torch.set_grad_enabled(self.finetune and torch.is_grad_enabled()):
            mask = torch.arange(waveform.shape[1], device=waveform.device)[None, :] < lengths[:, None]
            with torch.autocast("cuda", dtype=torch.float16, enabled=waveform.is_cuda):     # GPU 에서만 반정밀도
                hidden_states = self.wavlm(waveform, attention_mask=mask.long() if self.use_attention_mask else None,
                                           output_hidden_states=True).hidden_states
            frames = lengths
            for kernel, stride in zip(self.wavlm.config.conv_kernel, self.wavlm.config.conv_stride):
                frames = torch.div(frames - kernel, stride, rounding_mode="floor") + 1          # 샘플 길이 → 프레임 길이
            valid = (torch.arange(hidden_states[0].shape[1], device=waveform.device)[None, :] < frames[:, None])
            weight = valid.unsqueeze(-1).float()
            count = weight.sum(dim=1).clamp_min(1)
            stats = []
            for hidden in hidden_states:
                hidden = hidden.float()                                      # 분산은 float32 로 계산
                mean = (hidden * weight).sum(dim=1) / count
                variance = ((hidden - mean[:, None, :]).square() * weight).sum(dim=1) / count
                stats.append(torch.cat([mean, torch.sqrt(variance + 1e-5)], dim=-1))
            return torch.stack(stats, dim=1)

    def classify(self, stats):
        """층별 통계 → logit. 저장해 둔 통계로 분류기만 학습할 때도 이 함수를 쓴다."""
        z = (stats.float() - self.stat_mean) / self.stat_scale
        pooled = (z * torch.softmax(self.layer_logits, dim=0)[None, :, None]).sum(dim=1)
        return self.classifier(self.norm(pooled)).squeeze(-1)

    def forward(self, waveform, lengths):
        return self.classify(self.layer_stats(waveform, lengths))


def predict(model, rows, batch_size=32, device="cpu", num_workers=0):
    """rows 의 발화마다 신고자 확률. 반환: (확률 배열, 모델 forward 에 쓴 시간(초))."""
    model.eval()
    probabilities, model_seconds = np.zeros(len(rows)), 0.0
    loader = DataLoader(Segments(rows, model.do_normalize), batch_sampler=length_batches(rows, batch_size),
                        num_workers=num_workers, collate_fn=collate)
    with torch.inference_mode():
        for index, waveform, lengths in loader:
            waveform, lengths = waveform.to(device), lengths.to(device)
            if waveform.is_cuda:
                torch.cuda.synchronize()
            started = time.perf_counter()
            p = torch.sigmoid(model(waveform, lengths)).float()
            if waveform.is_cuda:
                torch.cuda.synchronize()
            model_seconds += time.perf_counter() - started
            probabilities[index] = p.cpu().numpy()
    return probabilities, model_seconds


# ============================================================
# 4. 체크포인트 · 파라미터 수
# ============================================================
def save_checkpoint(path, model, meta=None):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    torch.save({"format": FORMAT, "threshold": THRESHOLD, "config": model.wavlm.config.to_dict(),
                "do_normalize": model.do_normalize, "use_attention_mask": model.use_attention_mask,
                "finetuned": model.finetuned, "meta": meta or {},
                "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()}}, path + ".tmp")
    os.replace(path + ".tmp", path)             # 쓰다가 끊겨도 반쯤 쓴 파일이 남지 않게


def load_checkpoint(path, device="cpu"):
    """체크포인트만으로 모델 복원 (다운로드 없음). 반환: (모델, meta)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=True)      # tensor 와 기본 자료형만 읽음
    if ckpt.get("format") != FORMAT or float(ckpt["threshold"]) != THRESHOLD:
        raise ValueError("지원하지 않는 체크포인트입니다")
    model = RoleModel(WavLMConfig.from_dict(ckpt["config"]), ckpt["do_normalize"], ckpt["use_attention_mask"])
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.finetuned = bool(ckpt["finetuned"])
    return model.to(device).eval(), ckpt["meta"]


def count_parameters(model):
    """Total / Active / 학습한 파라미터 수. dense 모델이라 forward 에 모든 파라미터가 쓰인다 (Active = Total)."""
    total = sum(p.numel() for p in model.parameters())
    head = sum(p.numel() for p in model.head_parameters())
    tuned = sum(p.numel() for m in model.tuned_modules() for p in m.parameters()) if model.finetuned else 0
    return {"total": total, "active": total, "trained": head + tuned, "classifier": head,
            "wavlm_layers": model.wavlm.config.num_hidden_layers, "wavlm_finetuned": model.finetuned}

Writing /content/mission2/model.py


### train.py — 학습: 분류기 학습 → 미세조정 → 내부 dev 로 선택

In [ ]:
%%writefile "{PROJECT}/train.py"
"""
train.py — Mission 2 학습

    python train.py --train_audio_dir {wav 폴더} --train_label_dir {json 폴더} \
        --out_ckpt checkpoints/mission2.ckpt --work_dir {중간 결과 폴더}

절차 (Validation 폴더는 여기서 쓰지 않는다. 평가는 inference.py 로 따로 한다)
  1) 발화 목록 → 통화 단위로 내부 dev 를 떼고, 통화마다 발화 몇 개를 label 과 무관하게 고른다
     (한 통화의 발화는 같은 두 사람의 목소리라 서로 비슷하다. 통화당 몇 개만 쓰고 통화 수를 늘리는 편이 효율적)
  2) 1단계 — WavLM 동결: 발화마다 층별 통계를 한 번만 계산해 저장하고, 그 통계로 분류기만 학습
  3) 2단계 — 미세조정: 1단계 모델에서 시작해 WavLM Transformer 위쪽 2층과 분류기를 학습 (그 아래 층과 CNN 특징 추출기는 동결)
  4) 내부 dev Accuracy@0.5 가 가장 높은 모델(1단계 또는 2단계의 어느 epoch)을 체크포인트로 저장
모든 선택은 내부 dev 로만 하고, 판정 기준은 항상 확률 >= 0.5 다.
"""
import argparse
import hashlib
import json
import os
import platform
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

import model as M

HEAD = dict(epochs=40, batch_size=256, lr=1e-3, weight_decay=0.01, patience=6)                 # 1단계: 분류기
TUNE = dict(encoder_lr=1e-5, head_lr=1e-4, weight_decay=0.01, warmup_ratio=0.1, grad_clip=1.0)  # 2단계: 미세조정


# ------------------------------------------------------------
# 데이터: 통화 단위 분할, 통화당 발화 추출
# ------------------------------------------------------------
def split_calls(rows, dev_calls, seed):
    """통화를 무작위 순서로 섞어 앞의 dev_calls 개를 내부 dev 로 (같은 통화의 발화가 양쪽에 섞이지 않게)."""
    calls = sorted({r["call"] for r in rows})
    order = [calls[i] for i in np.random.default_rng(seed).permutation(len(calls))]
    dev = set(order[:dev_calls])
    return [r for r in rows if r["call"] not in dev], [r for r in rows if r["call"] in dev]


def sample_per_call(rows, per_call, seed):
    """통화마다 발화를 최대 per_call 개 고른다. 통화 이름으로 정한 난수를 써서 label·실행 순서와 무관하다."""
    by_call = {}
    for i, r in enumerate(rows):
        by_call.setdefault(r["call"], []).append(i)
    keep = []
    for call, index in by_call.items():
        rng = np.random.default_rng([seed, int(hashlib.sha256(call.encode()).hexdigest()[:8], 16)])
        index = sorted(index, key=lambda i: (rows[i]["a"], rows[i]["b"]))
        keep += [index[k] for k in rng.permutation(len(index))[:per_call]]
    return [rows[i] for i in sorted(keep)]


def scores(labels, probabilities):
    """Accuracy@0.5, log loss, 클래스별 Accuracy."""
    p = np.clip(probabilities, 1e-7, 1 - 1e-7)
    predicted = (probabilities >= M.THRESHOLD).astype(int)
    by_class = [round(float((predicted[labels == c] == c).mean()), 4) if (labels == c).any() else None for c in (0, 1)]
    return {"accuracy": float((predicted == labels).mean()),
            "loss": float(-(labels * np.log(p) + (1 - labels) * np.log(1 - p)).mean()),
            "acc_119": by_class[0], "acc_caller": by_class[1]}


# ------------------------------------------------------------
# 1단계: 동결 WavLM 통계 → 분류기 학습
# ------------------------------------------------------------
def frozen_stats(model, rows, args, device, name):
    """발화마다 층별 통계를 한 번 계산 (float16 으로 저장, 같은 발화 목록이면 파일에서 다시 읽음)."""
    key = hashlib.md5("\n".join(f"{r['call']}|{r['a']}|{r['b']}" for r in rows).encode()).hexdigest()[:12]
    path = os.path.join(args.work_dir, f"stats_{name}_{model.wavlm.config.num_hidden_layers}layers_{key}.npy")
    if os.path.exists(path):
        M.log(f"저장된 통계 사용: {path}")
        return np.load(path)
    started, stats = time.time(), None
    loader = DataLoader(M.Segments(rows, model.do_normalize), batch_sampler=M.length_batches(rows, args.batch_size),
                        num_workers=args.num_workers, collate_fn=M.collate)
    model.eval()
    for k, (index, waveform, lengths) in enumerate(loader):
        with torch.inference_mode():
            s = model.layer_stats(waveform.to(device), lengths.to(device)).to(torch.float16).cpu().numpy()
        if stats is None:
            stats = np.zeros((len(rows), *s.shape[1:]), dtype=np.float16)
        stats[index] = s
        if (k + 1) % 200 == 0:
            M.log(f"  통계 계산 {name}: batch {k + 1}/{len(loader)} ({time.time() - started:.0f}s)")
    np.save(path, stats)
    M.log(f"통계 계산 {name}: 발화 {len(rows):,}개, {time.time() - started:.0f}s ({len(rows) / (time.time() - started):.0f} utt/s)")
    return stats


def fit_head(model, x_train, y_train, x_dev, y_dev, device, seed):
    """저장한 통계로 분류기만 학습. 매 epoch 내부 dev Accuracy@0.5(같으면 loss)가 가장 좋은 상태를 남긴다."""
    torch.manual_seed(seed)
    x = torch.from_numpy(x_train).to(device)
    y = torch.from_numpy(y_train.astype(np.float32)).to(device)
    xd = torch.from_numpy(x_dev).to(device)
    # 층별 표준화 통계 (학습 데이터로만 계산). 거의 변하지 않는 차원은 1e-4 로 하한을 둔다
    mean = torch.stack([x[:, k].float().mean(dim=0) for k in range(x.shape[1])])
    var = torch.stack([x[:, k].float().var(dim=0, unbiased=False) for k in range(x.shape[1])])
    model.stat_mean.copy_(mean)
    model.stat_scale.copy_(torch.sqrt(var + 1e-4))
    optimizer = torch.optim.AdamW(model.head_parameters(), lr=HEAD["lr"], weight_decay=HEAD["weight_decay"])
    loss_fn = torch.nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(seed)
    history, best = [], None
    for epoch in range(1, HEAD["epochs"] + 1):
        model.train()
        total = 0.0
        for index in torch.randperm(len(x), generator=generator).split(HEAD["batch_size"]):
            index = index.to(device)
            loss = loss_fn(model.classify(x[index]), y[index])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total += float(loss.detach()) * len(index)
        model.eval()
        with torch.no_grad():
            p = torch.cat([torch.sigmoid(model.classify(xd[k:k + 4096])) for k in range(0, len(xd), 4096)]).cpu().numpy()
        record = {"epoch": epoch, "train_loss": total / len(x), **scores(y_dev, p.astype(np.float64))}
        history.append(record)
        if best is None or (record["accuracy"], -record["loss"]) > (best[0]["accuracy"], -best[0]["loss"]):
            best = (record, {k: v.clone() for k, v in model.state_dict().items() if not k.startswith("wavlm.")})
        elif epoch - best[0]["epoch"] >= HEAD["patience"]:
            break                                # dev 가 patience epoch 동안 나아지지 않으면 중단
    model.load_state_dict(best[1], strict=False)             # 분류기만 되돌림 (WavLM 은 그대로)
    M.log(f"1단계 분류기: best epoch {best[0]['epoch']} / dev Accuracy {best[0]['accuracy']:.4f} (저장한 통계 기준)")
    return history


# ------------------------------------------------------------
# 2단계: 미세조정
# ------------------------------------------------------------
def evaluate(model, rows, labels, args, device):
    """내부 dev 를 실제 추론 경로(M.predict)로 평가."""
    probabilities, _ = M.predict(model, rows, args.batch_size, device, args.num_workers)
    return scores(labels, probabilities)


def finetune(model, train_rows, dev_rows, y_dev, args, device, state, resuming):
    """WavLM Transformer 위쪽 2층 + 분류기 학습. 좋아질 때마다 체크포인트를 저장하고, 끊기면 last.pt 에서 이어 간다."""
    model.set_finetune(True)
    encoder_parameters = [p for m in model.tuned_modules() for p in m.parameters()]
    optimizer = torch.optim.AdamW([{"params": encoder_parameters, "lr": TUNE["encoder_lr"]},
                                   {"params": model.head_parameters(), "lr": TUNE["head_lr"]}],
                                  weight_decay=TUNE["weight_decay"])
    base_rates = [TUNE["encoder_lr"], TUNE["head_lr"]]
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    steps = len(M.length_batches(train_rows, args.batch_size))
    total, warmup = steps * args.finetune_epochs, int(steps * args.finetune_epochs * TUNE["warmup_ratio"])
    last = os.path.join(args.work_dir, "last.pt")
    if resuming:                                # 이어서 학습: 마지막으로 끝난 epoch 의 상태 복원
        resume = torch.load(last, map_location="cpu", weights_only=True)
        model.load_state_dict(resume["model"])
        optimizer.load_state_dict(resume["optimizer"])
        scaler.load_state_dict(resume["scaler"])
        M.log(f"epoch {len(state['tune_history'])} 다음부터 이어서 학습")
    else:
        torch.save({"state": state, "model": {k: v.cpu() for k, v in model.state_dict().items()},
                    "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict()}, last)
    labels = torch.tensor([r["speaker"] for r in train_rows], dtype=torch.float32)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    for epoch in range(len(state["tune_history"]) + 1, args.finetune_epochs + 1):
        started = time.time()
        torch.manual_seed(args.seed * 1000 + epoch)
        rng = np.random.default_rng([args.seed, epoch])
        loader = DataLoader(M.Segments(train_rows, model.do_normalize, crop_seed=args.seed * 1000 + epoch),
                            batch_sampler=M.length_batches(train_rows, args.batch_size, rng),
                            num_workers=args.num_workers, collate_fn=M.collate)
        model.train()
        running, seen = 0.0, 0
        for k, (index, waveform, lengths) in enumerate(loader):
            step = (epoch - 1) * steps + k       # warmup 동안 선형 증가, 이후 끝까지 선형 감소
            factor = (step + 1) / max(1, warmup) if step < warmup else max(0.0, (total - step) / max(1, total - warmup))
            for group, rate in zip(optimizer.param_groups, base_rates):
                group["lr"] = rate * factor
            logits = model(waveform.to(device), lengths.to(device))
            loss = loss_fn(logits.float(), labels[index].to(device))
            if not torch.isfinite(loss):
                raise RuntimeError(f"loss 가 발산했습니다 (epoch {epoch}, step {k})")
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(encoder_parameters + model.head_parameters(), TUNE["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.detach()) * len(index)
            seen += len(index)
            if (k + 1) % 200 == 0:
                M.log(f"  epoch {epoch} step {k + 1}/{steps} loss {running / seen:.4f} ({seen / (time.time() - started):.0f} utt/s)")
        record = {"epoch": epoch, "train_loss": running / seen, **evaluate(model, dev_rows, y_dev, args, device),
                  "seconds": round(time.time() - started, 1)}
        state["tune_history"].append(record)
        M.log(f"2단계 epoch {epoch}: train loss {record['train_loss']:.4f} / dev Accuracy {record['accuracy']:.4f} "
              f"(119대원 {record['acc_119']}, 신고자 {record['acc_caller']})")
        keep_best(model, record, f"미세조정 epoch {epoch}", args, state)
        # 재개용 파일: 학습 기록(state)도 함께 넣어 모델 상태와 항상 같은 epoch 를 가리키게 한다
        torch.save({"state": state, "model": {k: v.cpu() for k, v in model.state_dict().items()},
                    "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict()}, last + ".tmp")
        os.replace(last + ".tmp", last)
        save_state(args, state)
    os.remove(last)


def keep_best(model, record, name, args, state):
    """내부 dev Accuracy@0.5 가 지금까지보다 높으면(같으면 loss 가 낮으면) 체크포인트로 저장."""
    best = state["best"]
    if best is None or (record["accuracy"], -record["loss"]) > (best["accuracy"], -best["loss"]):
        state["best"] = {"model": name, "accuracy": record["accuracy"], "loss": record["loss"]}
        M.save_checkpoint(args.out_ckpt, model, {**state["meta"], "selected": name, "dev_accuracy": record["accuracy"]})


def save_state(args, state):
    with open(os.path.join(args.work_dir, "train_log.json"), "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_audio_dir", required=True)
    ap.add_argument("--train_label_dir", required=True)
    ap.add_argument("--out_ckpt", default="checkpoints/mission2.ckpt")
    ap.add_argument("--work_dir", default="work", help="통계 캐시·재개 파일·학습 기록을 두는 폴더")
    ap.add_argument("--pretrained", default=M.PRETRAINED, help="Hugging Face 이름 또는 로컬 폴더")
    ap.add_argument("--layers", type=int, default=4, help="사용할 WavLM Transformer 층 수")
    ap.add_argument("--dev_calls", type=int, default=1000, help="내부 dev 로 떼는 통화 수")
    ap.add_argument("--utts_per_call", type=int, default=6, help="학습 통화당 발화 수")
    ap.add_argument("--dev_utts_per_call", type=int, default=12, help="dev 통화당 발화 수")
    ap.add_argument("--finetune_epochs", type=int, default=4, help="0 이면 1단계(동결)만")
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--retrain", action="store_true", help="끝난 학습·끊긴 학습 기록을 무시하고 처음부터")
    args = ap.parse_args()
    os.makedirs(args.work_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    started = time.time()

    # 1) 데이터
    rows = M.build_manifest(args.train_label_dir, args.train_audio_dir, with_label=True)
    train_rows, dev_rows = split_calls(rows, args.dev_calls, args.seed)
    train_rows = sample_per_call(train_rows, args.utts_per_call, args.seed)
    dev_rows = sample_per_call(dev_rows, args.dev_utts_per_call, args.seed)
    if {r["call"] for r in train_rows} & {r["call"] for r in dev_rows} or not train_rows or not dev_rows:
        raise RuntimeError("train/dev 분할이 잘못되었습니다")
    y_train = np.array([r["speaker"] for r in train_rows])
    y_dev = np.array([r["speaker"] for r in dev_rows])
    for name, part, y in (("train", train_rows, y_train), ("dev", dev_rows, y_dev)):
        seconds = np.array([(r["b"] - r["a"]) / r["sr"] for r in part])
        if not 0.05 < np.median(seconds) < 30:           # 발화 길이가 비정상이면 startAt/endAt 단위(ms)가 다른 것
            raise RuntimeError(f"발화 길이 중앙값이 {np.median(seconds):.3f}초입니다. startAt/endAt 단위를 확인하세요")
        M.log(f"{name}: 통화 {len({r['call'] for r in part}):,}개 / 발화 {len(part):,}개 / 신고자 비율 {y.mean():.3f} / "
              f"길이 중앙값 {np.median(seconds):.2f}s / 6초 초과 {(seconds > 6).mean():.1%}")

    meta = {"pretrained": args.pretrained, "layers": args.layers, "seed": args.seed,
            "train_calls": len({r["call"] for r in train_rows}), "train_utterances": len(train_rows),
            "dev_calls": len({r["call"] for r in dev_rows}), "dev_utterances": len(dev_rows)}
    settings = {k: v for k, v in vars(args).items() if k not in ("retrain", "num_workers")}
    signature = hashlib.md5(json.dumps([meta, settings, HEAD, TUNE], sort_keys=True, default=str).encode()).hexdigest()
    log_path, last = os.path.join(args.work_dir, "train_log.json"), os.path.join(args.work_dir, "last.pt")
    if os.path.exists(log_path) and os.path.exists(args.out_ckpt) and not args.retrain:
        done = json.load(open(log_path, encoding="utf-8"))
        if done.get("signature") == signature and "train_minutes" in done:
            M.log(f"같은 설정의 학습이 이미 끝났습니다: {args.out_ckpt} (다시 학습하려면 --retrain)")
            return
    state = {"signature": signature, "meta": meta, "head": HEAD, "tune": TUNE, "head_history": [],
             "frozen": None, "tune_history": [], "best": None}
    resuming = False
    if os.path.exists(last) and not args.retrain:        # 미세조정 도중 끊긴 실행: 그 지점의 기록에서 이어 감
        saved = torch.load(last, map_location="cpu", weights_only=True)["state"]
        if saved["signature"] == signature:
            state, resuming = saved, True

    # 2) 1단계: 동결 WavLM 통계로 분류기 학습
    torch.manual_seed(args.seed)                 # 분류기 초기값까지 재현되도록
    model = M.RoleModel.from_pretrained(args.pretrained, args.layers).to(device)
    M.log(f"device {device} / WavLM {args.layers}층 / 파라미터 {M.count_parameters(model)['total']:,}")
    if not resuming:
        x_train = frozen_stats(model, train_rows, args, device, "train")
        x_dev = frozen_stats(model, dev_rows, args, device, "dev")
        state["head_history"] = fit_head(model, x_train, y_train, x_dev, y_dev, device, args.seed)
        del x_train, x_dev
        state["frozen"] = evaluate(model, dev_rows, y_dev, args, device)        # 실제 추론 경로로 다시 평가
        M.log(f"1단계(동결) dev Accuracy {state['frozen']['accuracy']:.4f} "
              f"(119대원 {state['frozen']['acc_119']}, 신고자 {state['frozen']['acc_caller']})")
        keep_best(model, state["frozen"], "동결", args, state)
        save_state(args, state)

    # 3) 2단계: 미세조정
    if args.finetune_epochs > 0:
        finetune(model, train_rows, dev_rows, y_dev, args, device, state, resuming)

    # 4) 요약
    best_model, _ = M.load_checkpoint(args.out_ckpt)
    state.update(parameters=M.count_parameters(best_model), train_minutes=round((time.time() - started) / 60, 1),
                 environment={"gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
                              "cpu_count": os.cpu_count(), "python": platform.python_version(), "torch": torch.__version__},
                 checkpoint_mb=round(os.path.getsize(args.out_ckpt) / 1e6, 1))
    save_state(args, state)
    print("\n단계 | epoch | train loss | dev Accuracy | dev loss")
    print(f"1 동결 | - | - | {state['frozen']['accuracy']:.4f} | {state['frozen']['loss']:.4f}")
    for h in state["tune_history"]:
        print(f"2 미세조정 | {h['epoch']} | {h['train_loss']:.4f} | {h['accuracy']:.4f} | {h['loss']:.4f}")
    M.log(f"저장: {args.out_ckpt} ({state['checkpoint_mb']}MB) / 선택 {state['best']['model']} / "
          f"dev Accuracy {state['best']['accuracy']:.4f} / 전체 {state['train_minutes']}분")


if __name__ == "__main__":
    main()

Writing /content/mission2/train.py


### inference.py — 제출 추론 (명령 1회로 CSV 생성, 오프라인)

In [ ]:
%%writefile "{PROJECT}/inference.py"
"""
inference.py — Mission 2 추론 (신고자 = 1 / 119대원 = 0)

    python inference.py --audio_dir {wav folder} --label_dir {json folder} \
        --ckpt_path checkpoints/mission2.ckpt --output ./outputs/mission2.csv

- 라벨 json 에서는 utterances 의 startAt, endAt 만 읽는다 (speaker, text, 발화 순서는 쓰지 않음)
- 발화마다 독립적으로 예측하고, 확률 >= 0.5 이면 신고자(1)
- 입력 발화마다 1행을 출력한다. 구간이 wav 길이를 넘으면 wav 끝에 맞춰 자르고,
  wav 가 없거나 구간을 읽을 수 없는 발화도 빼지 않고 신고자(1, 다수 클래스)로 채운다
- 출력 CSV 열: audio file name, startAt, endAt, speaker (startAt/endAt 은 입력 값 그대로)
- WavLM 설정·가중치가 체크포인트 안에 있어 인터넷 없이 실행된다
"""
import argparse
import csv
import json
import os
import platform
import time

os.environ.setdefault("HF_HUB_OFFLINE", "1")           # 인터넷 접속 시도 자체를 막음
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import numpy as np
import torch

import model as M


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    ap = argparse.ArgumentParser()
    ap.add_argument("--audio_dir", required=True, help="wav 폴더 (하위 폴더 포함)")
    ap.add_argument("--label_dir", required=True, help="라벨 json 폴더 (하위 폴더 포함)")
    ap.add_argument("--ckpt_path", default=os.path.join(here, "checkpoints", "mission2.ckpt"))
    ap.add_argument("--output", default="./outputs/mission2.csv")
    ap.add_argument("--batch_size", type=int, default=32, help="forward 1회에 넣는 발화 수")
    ap.add_argument("--num_workers", type=int, default=0, help="wav 읽기 프로세스 수")
    ap.add_argument("--device", default=None, help="cuda / cpu (기본: 자동)")
    ap.add_argument("--report", default=None, help="(선택) 추론 시간·파라미터 수·환경을 기록할 json 경로")
    args = ap.parse_args()
    started = time.time()
    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")

    model, meta = M.load_checkpoint(args.ckpt_path, device)
    rows = M.build_manifest(args.label_dir, args.audio_dir, with_label=False)     # startAt, endAt 만 읽음

    labels = np.ones(len(rows), dtype=int)               # 읽을 수 없는 발화는 신고자(1)로 채움
    usable = [i for i, r in enumerate(rows) if r["ok"]]
    infer_started, model_seconds = time.time(), 0.0
    if usable:
        probabilities, model_seconds = M.predict(model, [rows[i] for i in usable], args.batch_size, device, args.num_workers)
        labels[usable] = (probabilities >= M.THRESHOLD).astype(int)
    infer_seconds = time.time() - infer_started

    os.makedirs(os.path.dirname(os.path.abspath(args.output)), exist_ok=True)
    with open(args.output, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["audio file name", "startAt", "endAt", "speaker"])
        for row, label in zip(rows, labels):
            writer.writerow([row["wav_name"], row["start"], row["end"], int(label)])

    n = max(len(usable), 1)
    M.log(f"저장: {args.output} ({len(rows):,}행, 읽을 수 없어 1로 채운 발화 {len(rows) - len(usable):,}개)")
    M.log(f"추론 시간: {infer_seconds:.1f}s (발화당 {infer_seconds / n * 1000:.2f}ms, 그중 모델 forward {model_seconds / n * 1000:.2f}ms), "
          f"batch size {args.batch_size}, device {device}, 모델 로드 포함 전체 {time.time() - started:.1f}s")
    if args.report:
        with open(args.report, "w", encoding="utf-8") as f:
            json.dump({"utterances": len(rows), "unreadable_utterances": len(rows) - len(usable),
                       "batch_size": args.batch_size, "num_workers": args.num_workers, "device": device,
                       "inference_seconds": round(infer_seconds, 2), "ms_per_utterance": round(infer_seconds / n * 1000, 3),
                       "model_seconds": round(model_seconds, 2), "model_ms_per_utterance": round(model_seconds / n * 1000, 3),
                       "total_seconds": round(time.time() - started, 2), "parameters": M.count_parameters(model),
                       "gpu": torch.cuda.get_device_name(0) if device.startswith("cuda") else None,
                       "cpu_count": os.cpu_count(), "python": platform.python_version(), "torch": torch.__version__,
                       "checkpoint_meta": meta}, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    main()

Writing /content/mission2/inference.py


### prepare_data.py — 대회 제공 zip 에서 학습·평가 통화 꺼내기 (학습 준비 전용)

In [ ]:
%%writefile "{PROJECT}/prepare_data.py"
"""
prepare_data.py — 대회 제공 zip 에서 학습·평가 통화의 wav·json 을 꺼낸다 (학습 준비 전용)

    python prepare_data.py --zip_root {zip 파일 또는 zip 이 있는 폴더} --out_dir {꺼낼 폴더} --train_calls 29200 --val_calls 3640

zip 안의 구조 (대학부_전체데이터_Training_Validation.zip)
    대학부 데이터/Training/1.원천데이터/TS_서울_구급/*.wav      대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/*.json
    대학부 데이터/Validation/1.원천데이터/VS_서울_구급/*.wav    대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/*.json

- Training: 경로에 '서울'이 있는 파일만 사용 (Training 은 서울 데이터만)
- Validation: 평가 전용이며 학습에는 쓰지 않는다
- 라벨 json 과 이름이 같은 wav 가 있는 통화만 대상. 요청한 수가 전체보다 적으면 무작위(seed 고정)로 고른다. 라벨 내용은 읽지 않는다
- 이미 꺼낸 파일은 건너뛴다
결과: {out_dir}/train/wav, train/json, validation/wav, validation/json, selection.json
"""
import argparse
import json
import os
import time
import unicodedata
import zipfile

import numpy as np


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def nfc(text):
    """한글 이름이 자모가 분리된 형태(NFD)로 올 수 있어, 비교·출력 전에 NFC 로 맞춘다."""
    return unicodedata.normalize("NFC", text)


def entry_name(info):
    """zip 항목의 경로. Windows 에서 묶은 zip 은 한글 이름이 cp949 로 들어 있어 복원한다."""
    name = info.filename
    if not info.flag_bits & 0x800:
        try:
            name = name.encode("cp437").decode("cp949")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return nfc(name.replace("\\", "/"))


def split_of(folders):
    """경로의 폴더 이름으로 Training / Validation 을 구분한다."""
    for folder in folders:
        if folder == "Training" or folder.startswith(("TS_", "TL_")):
            return "train"
        if folder == "Validation" or folder.startswith(("VS_", "VL_")):
            return "validation"
    return None


def find_zips(zip_root):
    """zip 파일 하나를 직접 주거나, 폴더를 주면 그 아래의 zip 을 모두 찾는다."""
    if os.path.isfile(zip_root):
        return [zip_root]
    found = []
    for folder, subfolders, files in os.walk(zip_root):
        subfolders[:] = [d for d in subfolders if not d.startswith(".")]
        found += [os.path.join(folder, name) for name in files if name.lower().endswith(".zip")]
    return sorted(found)


def extract(archive, info, target):
    """항목 하나를 target 으로 꺼낸다. 같은 크기로 이미 있으면 건너뛴다."""
    if os.path.exists(target) and os.path.getsize(target) == info.file_size:
        return False
    with archive.open(info) as source, open(target + ".tmp", "wb") as out:
        while True:
            block = source.read(1 << 20)
            if not block:
                break
            out.write(block)
    os.replace(target + ".tmp", target)          # 중간에 끊겨도 반쯤 쓴 파일이 남지 않게
    return True


def prepare(zip_root, out_dir, train_calls, val_calls, seed=42):
    archives = [zipfile.ZipFile(path) for path in find_zips(zip_root)]
    items = {split: {".wav": {}, ".json": {}} for split in ("train", "validation")}
    not_seoul = 0
    for number, archive in enumerate(archives):
        for info in archive.infolist():
            name = entry_name(info)
            *folders, base = name.split("/")
            extension = os.path.splitext(base)[1].lower()
            split = split_of(folders)
            if split is None or extension not in (".wav", ".json") or base.startswith(".") or "__MACOSX" in folders:
                continue
            if split == "train" and "서울" not in name:
                not_seoul += 1
                continue
            items[split][extension].setdefault(base[:-len(extension)], (number, info))
    if not_seoul:
        log(f"[train] 서울이 아닌 파일 {not_seoul:,}개는 쓰지 않음")

    summary = {"seed": seed, "zips": [nfc(os.path.basename(a.filename)) for a in archives]}
    for number, (split, wanted) in enumerate((("train", train_calls), ("validation", val_calls))):
        wavs, labels = items[split][".wav"], items[split][".json"]
        names = sorted(set(wavs) & set(labels))                      # 같은 이름 1:1
        log(f"[{split}] 라벨 {len(labels):,} / wav {len(wavs):,} / 같은 이름 {len(names):,}")
        if not names:
            raise FileNotFoundError(f"{zip_root} 에서 {split} 의 wav·json 짝을 찾지 못했습니다")
        order = np.random.default_rng([seed, number]).permutation(len(names))
        chosen = sorted(names[i] for i in order[:wanted])
        log(f"[{split}] 통화 {len(names):,}개 중 {len(chosen):,}개 선택 (seed {seed})")
        wav_dir, json_dir = os.path.join(out_dir, split, "wav"), os.path.join(out_dir, split, "json")
        os.makedirs(wav_dir, exist_ok=True)
        os.makedirs(json_dir, exist_ok=True)
        targets = [(wavs[c], os.path.join(wav_dir, c + ".wav")) for c in chosen] \
            + [(labels[c], os.path.join(json_dir, c + ".json")) for c in chosen]
        targets.sort(key=lambda t: (t[0][0], t[0][1].header_offset))  # zip 안에 저장된 순서대로 읽는다
        started, written = time.time(), 0
        for k, ((archive_number, info), target) in enumerate(targets):
            written += extract(archives[archive_number], info, target)
            if (k + 1) % 5000 == 0 or k + 1 == len(targets):
                elapsed = time.time() - started
                log(f"  [{split}] {k + 1:,}/{len(targets):,} 파일 ({elapsed:.0f}s, 남은 예상 {elapsed / (k + 1) * (len(targets) - k - 1):.0f}s)")
        summary[split] = {"available_calls": len(names), "selected_calls": len(chosen), "newly_extracted": written,
                          "calls": chosen}
    overlap = set(summary["train"]["calls"]) & set(summary["validation"]["calls"])
    log(f"Training 과 Validation 에 함께 있는 통화: {len(overlap):,}개")
    for archive in archives:
        archive.close()
    with open(os.path.join(out_dir, "selection.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False)
    return summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip_root", required=True, help="대회 제공 zip 파일, 또는 zip 이 있는 폴더 (하위 폴더 포함)")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--train_calls", type=int, default=29200, help="Training 에서 꺼낼 통화 수 (학습 + 내부 dev)")
    ap.add_argument("--val_calls", type=int, default=3640, help="Validation 에서 꺼낼 통화 수")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()
    prepare(args.zip_root, args.out_dir, args.train_calls, args.val_calls, args.seed)

Writing /content/mission2/prepare_data.py


### README.md — 제출물 설명

In [ ]:
# Mission 2 신고자 / 119대원 발화 분류

발화 조각 하나(라벨 json의 `startAt`~`endAt` 구간)를 듣고 신고자(1)인지 119대원(0)인지 맞히는 문제. 평가 지표는 Accuracy.

## 실행

```bash
pip install -r requirements.txt
python inference.py --audio_dir {wav folder} --label_dir {json folder} \
    --ckpt_path checkpoints/mission2.ckpt --output ./outputs/mission2.csv
```

- 위 명령 하나로 발화 목록 생성, 추론, CSV 저장까지 돌아감
- 출력 열: `audio file name, startAt, endAt, speaker` (입력 발화 1개당 1행)
- 라벨 json에서 읽는 값은 `startAt`, `endAt` 두 개뿐. `speaker`, `text`, 발화 순서는 추론에 안 씀
- 확률 0.5 이상이면 신고자(1). threshold는 0.5 고정
- WavLM 설정과 가중치를 체크포인트에 같이 넣어 둬서 인터넷 없이 실행 가능. GPU 있으면 자동으로 사용하고, CPU로 돌리려면 `--device cpu`
- 구간이 wav 길이보다 길면 wav 끝에서 자름. wav가 없거나 구간을 못 읽는 발화는 신고자(1)로 채워서 행 수를 맞춤
- 선택 인자: `--batch_size 32`, `--num_workers 0`, `--report report.json` (추론 시간, 파라미터 수, 환경 기록)

## 모델

```text
발화 구간 (8kHz → 16kHz, 최대 6초, 0.25초 단위 zero padding)
  → WavLM-base-plus CNN + Transformer 앞 4층
  → 층별(임베딩 + 4층) 평균, 표준편차 1536차원 (패딩 구간 제외)
  → 층별 표준화 → softmax 층 가중합 → LayerNorm → Linear 256 → GELU → Dropout → Linear 1 → sigmoid
```

- 신고자는 휴대폰이나 현장 소음이 섞인 채널, 119대원은 상황실 헤드셋이라 채널과 음색 차이가 큼. 이런 정보는 WavLM 아래층에 주로 남아 있어서 12층 중 앞 4층만 사용
- 6초 넘는 발화는 가운데 6초만 사용
- padding 길이를 그 발화 길이로만 정하기 때문에 batch를 어떻게 묶든 결과가 같음

## 학습 (`train.py`)

1. Training 통화를 통화 단위로 나눠서 내부 dev를 따로 뺌. 학습용은 통화마다 발화를 몇 개씩만 뽑음(label 안 보고 랜덤)
2. 1단계, WavLM 동결: 발화별 층 통계를 한 번만 계산해 저장해 두고 분류기만 학습
3. 2단계, 미세조정: 1단계 모델에서 이어서 Transformer 위쪽 2층 + 분류기 학습. 그 아래 층과 CNN은 동결
4. 내부 dev Accuracy(@0.5)가 제일 높은 모델을 체크포인트로 저장

Validation 폴더는 학습이나 모델 선택에 안 쓰고 `inference.py`로 평가만 했음. Training은 서울 데이터만 사용.

```bash
python prepare_data.py --zip_root {대회 제공 zip} --out_dir {D} --train_calls 29200 --val_calls 3640
python train.py --train_audio_dir {D}/train/wav --train_label_dir {D}/train/json \
    --out_ckpt checkpoints/mission2.ckpt --work_dir work \
    --dev_calls 1000 --utts_per_call 12 --dev_utts_per_call 1000000 --finetune_epochs 4
```

실제로 돌린 명령은 `logs/prepare_log.txt`, `logs/train_log.txt`에서 `$`로 시작하는 줄에 있음.

## 파일

| 파일 | 내용 |
|---|---|
| `inference.py` | 제출용 추론 (명령 1회로 CSV 생성) |
| `model.py` | 발화 목록 생성, 음성 구간 처리, 모델, 예측, 체크포인트 저장/로드 |
| `train.py` | 학습 (분류기 학습 → 미세조정 → 내부 dev 기준 선택) |
| `prepare_data.py` | 대회 제공 zip에서 학습/평가 통화 추출 |
| `checkpoints/mission2.ckpt` | 학습된 모델 |
| `logs/` | 학습 로그, 추론 로그, 효율 기록(`efficiency_report.md`) |
| `requirements.txt` | 패키지 목록 |


Writing /content/mission2/README.md


## 2. 환경 준비 (런타임: GPU)

In [ ]:
for name in ['model.py', 'train.py', 'inference.py', 'prepare_data.py']:      # 저장한 코드의 문법 확인
    compile((PROJECT/name).read_text(encoding='utf-8'), name, 'exec')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')

from google.colab import drive
drive.mount('/content/drive')

torch 2.11.0+cu130 | GPU: NVIDIA A100-SXM4-40GB
Mounted at /content/drive


## 3. 설정

- `ZIP_ROOT`: 대회 제공 zip(`대학부_전체데이터_Training_Validation.zip`). zip이 있는 폴더를 줘도 됩니다.
- `DATA`: 꺼낸 wav·json을 두는 로컬 폴더. `RUN`: 체크포인트·로그·통계 캐시를 두는 Drive 폴더. **실험마다 다른 폴더**를 써야 로그와 캐시가 섞이지 않습니다.
- `BASELINE_RUN`: 6번에서 이번 실행과 비교할 기준 실행의 폴더. 없으면 `None`.
- 학습 규모: 대회 제공 Training 29,200통화를 학습 `TRAIN_CALLS` + 내부 dev `DEV_CALLS`로 통화 단위로 나눕니다. 학습은 통화마다 `UTTS_PER_CALL`발화를 쓰고, 내부 dev는 모든 발화로 평가합니다.
- `UTTS_PER_CALL`을 바꾸면 README의 `train.py` 명령도 같은 값으로 고칩니다.

In [ ]:
ZIP_ROOT = next(p for p in Path('/content/drive/MyDrive').glob('*.zip') if 'Training_Validation' in p.name)   # 대회 제공 zip
RUN = Path('/content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u12')            # 실험마다 다른 폴더
BASELINE_RUN = Path('/content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u6')    # 6번에서 비교할 기준 실행 (없으면 None)

TRAIN_CALLS, DEV_CALLS = 28200, 1000     # 대회 제공 Training 29,200통화 = 학습 28,200 + 내부 dev 1,000
UTTS_PER_CALL, DEV_UTTS_PER_CALL = 12, 10**6   # 학습은 통화당 12발화 (기준 실행은 6), 내부 dev 는 모든 발화로 평가
VAL_CALLS = 3640                         # 공식 Validation 에서 평가할 통화 수 (모든 발화 평가)
LAYERS = 4                               # 사용할 WavLM Transformer 층 수
FINETUNE_EPOCHS = 4                      # 0 이면 미세조정 없이 분류기만 학습
BATCH_SIZE, NUM_WORKERS, SEED = 32, 2, 42

DATA = Path(f'/content/dcc_subset_tr{TRAIN_CALLS + DEV_CALLS}_va{VAL_CALLS}')
CKPT, WORK, LOGS = RUN/'mission2.ckpt', RUN/'work', RUN/'logs'
LOGS.mkdir(parents=True, exist_ok=True)

def run(*command, log_name, env=None, cwd=None):
    """명령을 별도 프로세스로 실행하고, 출력을 화면과 로그 파일에 함께 남긴다."""
    command = [str(c) for c in command]
    with open(LOGS/log_name, 'a', encoding='utf-8') as log:
        log.write('\n$ ' + ' '.join(command) + '\n')
        with subprocess.Popen(command, cwd=cwd or PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
        if process.returncode:
            raise RuntimeError(f'명령이 실패했습니다 (종료 코드 {process.returncode})')
print('데이터:', DATA, '| 결과:', RUN)

데이터: /content/dcc_subset_tr29200_va3640 | 결과: /content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u12


## 4. 데이터 준비 — zip에서 통화 꺼내기

Training(서울)에서 `TRAIN_CALLS + DEV_CALLS`통화, Validation에서 `VAL_CALLS`통화의 wav와 json을 꺼냅니다. 요청한 수가 전체보다 적을 때만 무작위(seed 고정)로 고릅니다.
같은 이름의 wav가 있는 통화만 대상이고, 라벨 내용은 읽지 않습니다. 이미 꺼낸 파일은 건너뜁니다.

In [ ]:
run(sys.executable, '-u', 'prepare_data.py', '--zip_root', ZIP_ROOT, '--out_dir', DATA,
    '--train_calls', TRAIN_CALLS + DEV_CALLS, '--val_calls', VAL_CALLS, '--seed', SEED, log_name='prepare_log.txt')
shutil.copy2(DATA/'selection.json', LOGS/'selection.json')       # 고른 통화 목록
for part in ['train', 'validation']:
    print(part, 'wav', len(list((DATA/part/'wav').glob('*.wav'))), '/ json', len(list((DATA/part/'json').glob('*.json'))))

[14:00:09] [train] 라벨 29,200 / wav 29,200 / 같은 이름 29,200
[14:00:09] [train] 통화 29,200개 중 29,200개 선택 (seed 42)
[14:00:44]   [train] 5,000/58,400 파일 (35s, 남은 예상 374s)
[14:01:28]   [train] 10,000/58,400 파일 (79s, 남은 예상 381s)
[14:02:06]   [train] 15,000/58,400 파일 (117s, 남은 예상 338s)
[14:02:39]   [train] 20,000/58,400 파일 (150s, 남은 예상 288s)
[14:03:21]   [train] 25,000/58,400 파일 (192s, 남은 예상 256s)
[14:04:10]   [train] 30,000/58,400 파일 (241s, 남은 예상 228s)
[14:05:02]   [train] 35,000/58,400 파일 (293s, 남은 예상 196s)
[14:05:49]   [train] 40,000/58,400 파일 (339s, 남은 예상 156s)
[14:06:36]   [train] 45,000/58,400 파일 (387s, 남은 예상 115s)
[14:07:29]   [train] 50,000/58,400 파일 (440s, 남은 예상 74s)
[14:08:29]   [train] 55,000/58,400 파일 (500s, 남은 예상 31s)
[14:08:58]   [train] 58,400/58,400 파일 (529s, 남은 예상 0s)
[14:08:58] [validation] 라벨 3,640 / wav 3,640 / 같은 이름 3,640
[14:08:58] [validation] 통화 3,640개 중 3,640개 선택 (seed 42)
[14:09:20]   [validation] 5,000/7,280 파일 (21s, 남은 예상 10s)
[14:09:35]   [validation] 7,280/7,280 파일

## 5. 학습

1. 통화 단위로 내부 dev를 떼고, 통화마다 발화를 label과 무관하게 고릅니다.
2. **1단계:** WavLM을 동결하고 발화마다 층별 통계를 한 번 계산해, 그 통계로 분류기만 학습합니다.
3. **2단계:** 1단계 모델에서 시작해 WavLM Transformer 위쪽 2층을 미세조정합니다(그 아래 층과 CNN은 동결).
4. 내부 dev Accuracy@0.5가 가장 높은 모델을 체크포인트로 저장합니다.

Validation은 여기서 쓰지 않습니다. 연결이 끊기면 이 셀을 다시 실행하면 끝난 epoch 다음부터 이어집니다. 로그는 `logs/train_log.txt`에도 남습니다.

In [ ]:
if not torch.cuda.is_available():
    print('경고: GPU 가 없어 매우 느립니다. 런타임 유형을 GPU 로 바꾸세요.')
run(sys.executable, '-u', 'train.py', '--train_audio_dir', DATA/'train/wav', '--train_label_dir', DATA/'train/json',
    '--out_ckpt', CKPT, '--work_dir', WORK, '--layers', LAYERS, '--dev_calls', DEV_CALLS,
    '--utts_per_call', UTTS_PER_CALL, '--dev_utts_per_call', DEV_UTTS_PER_CALL, '--finetune_epochs', FINETUNE_EPOCHS,
    '--batch_size', BATCH_SIZE, '--num_workers', NUM_WORKERS, '--seed', SEED, log_name='train_log.txt')
train_log = json.loads((WORK/'train_log.json').read_text(encoding='utf-8'))
shutil.copy2(WORK/'train_log.json', LOGS/'train_log.json')

I0000 00:00:1791641387.766347    5308 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791641388.917313    5308 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791641391.781058    5308 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791641391.781584    5308 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

PosixPath('/content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u12/logs/train_log.json')

## 6. 내부 dev 상세 진단 · 기준 실행과 비교 (분석 전용)

모델 선택에 쓰는 내부 dev(서울 Training에서 뗀 통화)만 봅니다. Validation은 쓰지 않고, 모델 입력·학습·제출 추론도 바꾸지 않습니다.

1. 체크포인트를 다시 읽어 내부 dev를 평가하고(학습 로그의 Accuracy와 같아야 함), 길이별·클래스별 Accuracy, 오류 방향, 확신도, 층 가중치를 표로 남깁니다.
2. 상대 화자 발화와 시간이 겹치는 비율별 Accuracy도 봅니다. 이 값은 다른 발화의 시간·정답을 읽으므로 **오류 분석에만** 쓰고 모델 입력으로는 쓰지 않습니다.
3. `BASELINE_RUN`의 체크포인트를 같은 발화로 평가해, Accuracy 차이와 통화 단위 bootstrap 95% 구간을 냅니다. 구간이 0을 포함하면 차이를 확인할 수 없는 것으로 봅니다.

결과는 `RUN/diagnostics/`에 저장하고 제출 폴더에는 넣지 않습니다.

In [ ]:
# 분석 전용: 내부 dev(서울 Training 에서 뗀 통화)의 정답으로 오류를 나눠 본다. 모델 입력·학습·제출 추론은 바꾸지 않는다
import csv, importlib
import numpy as np
sys.path.insert(0, str(PROJECT))
import model as M, train as T
M, T = importlib.reload(M), importlib.reload(T)                    # 위에서 저장한 코드를 그대로 사용

DIAG = RUN/'diagnostics'                                           # 제출 폴더에는 넣지 않는 분석 결과
DIAG.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 내부 dev 를 train.py 와 같은 함수·seed 로 다시 만든다 → 학습 때와 같은 통화·발화
all_rows = M.build_manifest(str(DATA/'train/json'), str(DATA/'train/wav'), with_label=True)
dev_pool = T.split_calls(all_rows, DEV_CALLS, SEED)[1]             # dev 통화의 모든 발화
dev_rows = T.sample_per_call(dev_pool, DEV_UTTS_PER_CALL, SEED)    # 그중 평가에 쓴 발화
y = np.array([r['speaker'] for r in dev_rows])
seconds = np.array([(r['b'] - r['a']) / r['sr'] for r in dev_rows])
dev_call = np.array([r['call'] for r in dev_rows])


def other_speaker_overlap(rows, pool):
    """발화 구간 중, 같은 통화의 상대 화자 발화와 시간이 겹치는 비율 (0~1).

    오류 원인을 나눠 보는 용도로만 쓴다. 다른 발화의 시간·정답을 읽으므로 모델 입력으로는 쓸 수 없다.
    """
    by_call = {}
    for r in pool:
        by_call.setdefault(r['call'], []).append((r['a'], r['b'], r['speaker']))
    for spans in by_call.values():
        spans.sort()
    ratio = np.zeros(len(rows))
    for i, r in enumerate(rows):
        covered, end = 0, r['a']
        for a, b, speaker in by_call[r['call']]:
            if speaker == r['speaker'] or a >= r['b']:
                continue
            a, b = max(a, end), min(b, r['b'])                     # 이미 센 부분은 다시 세지 않는다
            if b > a:
                covered, end = covered + b - a, b
        ratio[i] = covered / max(r['b'] - r['a'], 1)
    return ratio


def dev_probabilities(ckpt_path):
    """체크포인트를 읽어 내부 dev 발화마다 신고자 확률을 구한다 (제출 추론과 같은 M.predict)."""
    model, meta = M.load_checkpoint(str(ckpt_path), device)
    p, _ = M.predict(model, dev_rows, BATCH_SIZE, device, NUM_WORKERS)
    return p, model, meta


def share(mask, hit):
    return f'{hit[mask].mean():.4f}' if mask.any() else '-'


def breakdown(title, groups, p, total=None, total_name='전체'):
    """구간별 발화 수, 클래스 구성, Accuracy, 오류 방향(0=119대원, 1=신고자)."""
    predicted = (p >= M.THRESHOLD).astype(int)
    hit = predicted == y
    total = np.ones(len(y), dtype=bool) if total is None else total
    lines = [f'### {title}', '',
             '| 구간 | 발화 | 비중 | 실제 신고자 비율 | 예측 신고자 비율 | 평균 확률 | Accuracy | 119대원 Acc | 신고자 Acc | 0→1 오류 | 1→0 오류 |',
             '|---|---|---|---|---|---|---|---|---|---|---|']
    for name, m in list(groups) + [(total_name, total)]:
        if m.any():
            lines.append(f'| {name} | {m.sum():,} | {m.mean():.1%} | {y[m].mean():.3f} | {predicted[m].mean():.3f} | {p[m].mean():.3f} | '
                         f'{hit[m].mean():.4f} | {share(m & (y == 0), hit)} | {share(m & (y == 1), hit)} | '
                         f'{int((m & (y == 0) & ~hit).sum()):,} | {int((m & (y == 1) & ~hit).sum()):,} |')
    return lines + ['']


def by_confidence(p):
    """확신도 max(p, 1-p) 별 Accuracy. 높은 확신으로 틀린 발화가 많으면 입력만으로는 가르기 어려운 사례다."""
    confidence, hit = np.maximum(p, 1 - p), (p >= M.THRESHOLD).astype(int) == y
    lines = ['### 확신도별', '', '| 확신도 max(p, 1-p) | 발화 | 비중 | Accuracy | 전체 오류 중 비중 |', '|---|---|---|---|---|']
    for name, low, high in (('0.5-0.6', .5, .6), ('0.6-0.8', .6, .8), ('0.8-0.95', .8, .95), ('0.95-1', .95, 1.01)):
        m = (confidence >= low) & (confidence < high)
        if m.any():
            lines.append(f'| {name} | {m.sum():,} | {m.mean():.1%} | {hit[m].mean():.4f} | {(m & ~hit).sum() / max((~hit).sum(), 1):.1%} |')
    return lines + ['']


LENGTHS = [('<0.5s', seconds < .5), ('0.5-1s', (seconds >= .5) & (seconds < 1)), ('1-3s', (seconds >= 1) & (seconds < 3)),
           ('3-6s', (seconds >= 3) & (seconds < 6)), ('>=6s', seconds >= 6)]
overlap = other_speaker_overlap(dev_rows, dev_pool)
OVERLAPS = [('겹침 없음', overlap == 0), ('0-25%', (overlap > 0) & (overlap <= .25)),
            ('25-50%', (overlap > .25) & (overlap <= .5)), ('50% 초과', overlap > .5)]

p_dev, dev_model, dev_meta = dev_probabilities(CKPT)
recorded = json.loads((WORK/'train_log.json').read_text(encoding='utf-8'))['best']['accuracy']
dev_accuracy = float(((p_dev >= M.THRESHOLD).astype(int) == y).mean())
layer_weights = torch.softmax(dev_model.layer_logits.detach().float().cpu(), dim=0).tolist()
report = [f'# 내부 dev 상세 진단 — {RUN.name}', '',
          f'- 통화 {len(set(dev_call)):,}개 · 발화 {len(dev_rows):,}개 · Accuracy {dev_accuracy:.4f} (학습 로그 {recorded:.4f})',
          f'- 선택된 모델: {dev_meta.get("selected")} / attention mask 사용: {dev_model.use_attention_mask} / 입력 정규화: {dev_model.do_normalize}',
          '- 층 가중치 (임베딩, 1층, 2층, …): ' + ', '.join(f'{w:.3f}' for w in layer_weights), '']
del dev_model
if abs(dev_accuracy - recorded) > 5e-4:
    report += ['**경고: 학습 로그의 dev Accuracy 와 다릅니다. DATA·DEV_CALLS·DEV_UTTS_PER_CALL·SEED 가 학습 때와 같은지 확인하세요.**', '']
report += breakdown('길이별', LENGTHS, p_dev)
report += breakdown('상대 화자 발화와 겹치는 비율별', OVERLAPS, p_dev)
report += breakdown('길이별 — 겹침이 없는 발화만', [(name, m & (overlap == 0)) for name, m in LENGTHS], p_dev,
                    total=overlap == 0, total_name='겹침 없음 전체')
report += by_confidence(p_dev)

(DIAG/'dev_diagnostics.md').write_text('\n'.join(report), encoding='utf-8')
with open(DIAG/'dev_predictions.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['call', 'startAt', 'endAt', 'seconds', 'overlap', 'speaker', 'probability'])
    for r, s, o, p in zip(dev_rows, seconds, overlap, p_dev):
        writer.writerow([r['call'], r['start'], r['end'], f'{s:.3f}', f'{o:.3f}', r['speaker'], f'{p:.6f}'])
print('\n'.join(report))
print('저장:', DIAG/'dev_diagnostics.md', '·', DIAG/'dev_predictions.csv')

[14:47:08] 발화 목록: 통화 29,200개 / 발화 889,358개
# 내부 dev 상세 진단 — mission2_official_u12

- 통화 1,000개 · 발화 30,356개 · Accuracy 0.9209 (학습 로그 0.9209)
- 선택된 모델: 미세조정 epoch 4 / attention mask 사용: True / 입력 정규화: False
- 층 가중치 (임베딩, 1층, 2층, …): 0.257, 0.212, 0.144, 0.160, 0.228

### 길이별

| 구간 | 발화 | 비중 | 실제 신고자 비율 | 예측 신고자 비율 | 평균 확률 | Accuracy | 119대원 Acc | 신고자 Acc | 0→1 오류 | 1→0 오류 |
|---|---|---|---|---|---|---|---|---|---|---|
| <0.5s | 2,428 | 8.0% | 0.513 | 0.496 | 0.526 | 0.9110 | 0.9256 | 0.8972 | 88 | 128 |
| 0.5-1s | 7,312 | 24.1% | 0.548 | 0.612 | 0.556 | 0.8820 | 0.7986 | 0.9508 | 666 | 197 |
| 1-3s | 14,735 | 48.5% | 0.470 | 0.518 | 0.467 | 0.9193 | 0.8787 | 0.9651 | 947 | 242 |
| 3-6s | 4,753 | 15.7% | 0.592 | 0.608 | 0.598 | 0.9741 | 0.9489 | 0.9915 | 99 | 24 |
| >=6s | 1,128 | 3.7% | 0.729 | 0.733 | 0.728 | 0.9920 | 0.9771 | 0.9976 | 7 | 2 |
| 전체 | 30,356 | 100.0% | 0.521 | 0.561 | 0.523 | 0.9209 | 0.8757 | 0.9625 | 1,807 | 593 |

### 상대 화자 발화와 겹치는 비율별

| 구간 | 발화 | 비중 | 실제 신고자 비율 | 

In [ ]:
# 기준 실행(BASELINE_RUN)의 체크포인트를 같은 내부 dev 발화로 평가해, 이번 실행과 발화 단위로 비교한다
BASE_CKPT = Path(BASELINE_RUN)/'mission2.ckpt' if BASELINE_RUN else None
if BASE_CKPT is None or not BASE_CKPT.exists() or Path(BASELINE_RUN).resolve() == RUN.resolve():
    print('비교할 기준 실행이 없어 건너뜁니다. BASELINE_RUN =', BASELINE_RUN)
else:
    p_base, base_model, base_meta = dev_probabilities(BASE_CKPT)
    del base_model
    hit_new, hit_base = (p_dev >= M.THRESHOLD).astype(int) == y, (p_base >= M.THRESHOLD).astype(int) == y
    same_dev = (base_meta.get('seed') == SEED and base_meta.get('dev_calls') == len(set(dev_call))
                and base_meta.get('dev_utterances') == len(dev_rows)
                and abs(float(hit_base.mean()) - base_meta.get('dev_accuracy', -1)) <= 5e-4)
    if not same_dev:
        # dev 통화가 다르면 기준 모델이 이 통화로 학습했을 수 있어 비교가 성립하지 않는다
        print('기준 실행의 내부 dev 가 이번 실행과 달라 비교하지 않습니다.',
              {k: base_meta.get(k) for k in ('seed', 'dev_calls', 'dev_utterances', 'dev_accuracy')},
              f'/ 이번 dev 로 다시 잰 기준 Accuracy {hit_base.mean():.4f}')
    else:
        calls, inverse = np.unique(dev_call, return_inverse=True)

        def call_bootstrap(mask, n_boot=10000):
            """통화를 복원추출해 Accuracy 차이(이번 - 기준)의 95% 구간을 구한다.

            같은 통화의 발화는 같은 두 사람의 목소리라 서로 독립이 아니다. 발화가 아니라 통화를 뽑아야 구간이 좁게 나오지 않는다.
            """
            count = np.bincount(inverse[mask], minlength=len(calls)).astype(float)
            gain = np.bincount(inverse[mask], weights=hit_new[mask].astype(float) - hit_base[mask].astype(float),
                               minlength=len(calls))
            rng, deltas = np.random.default_rng(SEED), []
            for _ in range(n_boot // 1000):
                pick = rng.integers(0, len(calls), size=(1000, len(calls)))
                deltas.append(gain[pick].sum(axis=1) / np.maximum(count[pick].sum(axis=1), 1))
            deltas = np.concatenate(deltas)
            return np.percentile(deltas, 2.5), np.percentile(deltas, 97.5), float((deltas > 0).mean())

        lines = [f'# 내부 dev 비교 — 이번 {RUN.name} vs 기준 {Path(BASELINE_RUN).name}', '',
                 f'- 통화 {len(calls):,}개 · 발화 {len(y):,}개 (같은 발화) / 이번만 맞음 {int((hit_new & ~hit_base).sum()):,}개 · '
                 f'기준만 맞음 {int((hit_base & ~hit_new).sum()):,}개', '',
                 '| 구간 | 발화 | 기준 Acc | 이번 Acc | 차이(%p) | 95% 구간(%p, 통화 단위 bootstrap) | 이번이 높은 비율 |',
                 '|---|---|---|---|---|---|---|']
        everything = np.ones(len(y), dtype=bool)
        for name, m in [('전체', everything), ('119대원', y == 0), ('신고자', y == 1)] + LENGTHS:
            if m.any():
                low, high, wins = call_bootstrap(m)
                lines.append(f'| {name} | {m.sum():,} | {hit_base[m].mean():.4f} | {hit_new[m].mean():.4f} | '
                             f'{(hit_new[m].mean() - hit_base[m].mean()) * 100:+.2f} | {low * 100:+.2f} ~ {high * 100:+.2f} | {wins:.1%} |')
        low, high, _ = call_bootstrap(everything)
        verdict = ('이번 설정이 더 좋음 (구간 전체가 0보다 큼) → 이번 설정 채택' if low > 0 else
                   '기준이 더 좋음 (구간 전체가 0보다 작음) → 기준 유지' if high < 0 else
                   '차이를 확인할 수 없음 (구간이 0을 포함) → 더 단순한 기준 유지')
        lines += ['', f'**판정(전체 Accuracy): {verdict}**']
        (DIAG/f'compare_with_{Path(BASELINE_RUN).name}.md').write_text('\n'.join(lines), encoding='utf-8')
        print('\n'.join(lines))

# 내부 dev 비교 — 이번 mission2_official_u12 vs 기준 mission2_official_u6

- 통화 1,000개 · 발화 30,356개 (같은 발화) / 이번만 맞음 381개 · 기준만 맞음 240개

| 구간 | 발화 | 기준 Acc | 이번 Acc | 차이(%p) | 95% 구간(%p, 통화 단위 bootstrap) | 이번이 높은 비율 |
|---|---|---|---|---|---|---|
| 전체 | 30,356 | 0.9163 | 0.9209 | +0.46 | +0.30 ~ +0.63 | 100.0% |
| 119대원 | 14,542 | 0.8696 | 0.8757 | +0.61 | +0.37 ~ +0.86 | 100.0% |
| 신고자 | 15,814 | 0.9592 | 0.9625 | +0.33 | +0.10 ~ +0.56 | 99.8% |
| <0.5s | 2,428 | 0.9143 | 0.9110 | -0.33 | -1.12 ~ +0.44 | 19.1% |
| 0.5-1s | 7,312 | 0.8755 | 0.8820 | +0.64 | +0.21 ~ +1.08 | 99.8% |
| 1-3s | 14,735 | 0.9133 | 0.9193 | +0.60 | +0.40 ~ +0.81 | 100.0% |
| 3-6s | 4,753 | 0.9714 | 0.9741 | +0.27 | +0.08 ~ +0.48 | 99.7% |
| >=6s | 1,128 | 0.9920 | 0.9920 | +0.00 | +0.00 ~ +0.00 | 0.0% |

**판정(전체 Accuracy): 이번 설정이 더 좋음 (구간 전체가 0보다 큼) → 이번 설정 채택**


## 7. 제출 명령으로 Validation 추론 · 정확도 · 효율 기록

1. 제출 명령(`python inference.py --audio_dir --label_dir --ckpt_path --output`)을 별도 프로세스로 한 번 실행합니다.
2. 결과 CSV의 형식(열 이름, 행 수, startAt/endAt)과 정확도를 확인합니다. 정답은 이 셀에서만 읽고, `inference.py`는 speaker를 읽지 않습니다.
3. 파라미터 수, 실행 환경, batch size, 추론 시간을 `efficiency_report.md`에 저장합니다.

In [ ]:
import csv
import numpy as np
VAL_CSV, REPORT = RUN/'mission2_validation.csv', LOGS/'inference_report.json'
run(sys.executable, '-u', 'inference.py', '--audio_dir', DATA/'validation/wav', '--label_dir', DATA/'validation/json',
    '--ckpt_path', CKPT, '--output', VAL_CSV, '--batch_size', BATCH_SIZE, '--num_workers', NUM_WORKERS,
    '--report', REPORT, log_name='inference_log.txt')

# 형식·정확도 확인: json 파일 이름순 × 발화 순서 = inference.py 의 행 순서
with open(VAL_CSV, encoding='utf-8') as f:
    header, *lines = list(csv.reader(f))
truth = [(str(u.get('startAt')), str(u.get('endAt')), u.get('speaker'), (u['endAt'] - u['startAt']) / 1000)
         for path in sorted((DATA/'validation/json').glob('*.json'))
         for u in json.loads(path.read_text(encoding='utf-8-sig'))['utterances']]
assert header == ['audio file name', 'startAt', 'endAt', 'speaker'], '열 이름 오류'
assert len(lines) == len(truth), '행 수가 입력 발화 수와 다름'
assert all(line[1:3] == list(t[:2]) and line[3] in ('0', '1') for line, t in zip(lines, truth)), 'startAt/endAt 또는 speaker 값 오류'
y = np.array([t[2] for t in truth]); p = np.array([int(line[3]) for line in lines]); seconds = np.array([t[3] for t in truth])
accuracy = float((p == y).mean())
confusion = [[int(((y == a) & (p == b)).sum()) for b in (0, 1)] for a in (0, 1)]
by_length = {name: (int(m.sum()), round(float((p[m] == y[m]).mean()), 4) if m.any() else None)
             for name, m in (('<0.5s', seconds < .5), ('0.5-1s', (seconds >= .5) & (seconds < 1)), ('1-3s', (seconds >= 1) & (seconds < 3)),
                             ('3-6s', (seconds >= 3) & (seconds < 6)), ('>=6s', seconds >= 6))}
print(f'\n형식 확인 통과 | 통화 {len(list((DATA/"validation/json").glob("*.json"))):,}개 · 발화 {len(lines):,}개 | Validation Accuracy {accuracy:.4f}')
print(f'클래스별 Accuracy: 119대원 {confusion[0][0] / max(sum(confusion[0]), 1):.4f} / 신고자 {confusion[1][1] / max(sum(confusion[1]), 1):.4f}')
print('혼동행렬 (행=정답, 열=예측; 0,1 순서):', confusion)
print('길이별 (발화 수, Accuracy):', by_length)

# 효율 기록
r, t = json.loads(REPORT.read_text(encoding='utf-8')), train_log
c = r['parameters']
report = f"""# Mission 2 결과·효율 기록

## 모델
- WavLM-base-plus 앞 {c['wavlm_layers']}층의 층별 평균·표준편차 → 층 가중합 → 분류기 (선택된 모델: {t['best']['model']})
- 학습: 서울 Training {t['meta']['train_calls']:,}통화 · {t['meta']['train_utterances']:,}발화 (내부 dev {t['meta']['dev_calls']:,}통화 · {t['meta']['dev_utterances']:,}발화)
- 내부 dev Accuracy {t['best']['accuracy']:.4f} / Validation Accuracy {accuracy:.4f} ({len(lines):,}발화), threshold 0.5 고정
- 클래스별 Validation Accuracy: 119대원 {confusion[0][0] / max(sum(confusion[0]), 1):.4f} / 신고자 {confusion[1][1] / max(sum(confusion[1]), 1):.4f}

## 파라미터 (dense 모델이라 Total = Active)
- Total {c['total']:,} / Active {c['active']:,}
- 이 데이터로 학습한 파라미터 {c['trained']:,} (분류기 {c['classifier']:,}{' + WavLM Transformer 위쪽 2층' if c['wavlm_finetuned'] else ', WavLM 은 동결'})
- 체크포인트 {CKPT.stat().st_size / 1e6:.0f}MB

## 실행 환경
- GPU {r['gpu']} / CPU 코어 {r['cpu_count']} / Python {r['python']} / torch {r['torch']}
- 학습 시간 {t['train_minutes']}분 (학습 환경 GPU {t['environment']['gpu']})

## Validation 추론 (inference.py 1회 실행)
- batch size {r['batch_size']} (padding 후 길이가 같은 발화끼리), wav 읽기 프로세스 {r['num_workers']}개
- 발화 {r['utterances']:,}개: {r['inference_seconds']}초, 발화당 평균 {r['ms_per_utterance']}ms (wav 읽기·전처리·모델 포함)
- 그중 모델 forward: {r['model_seconds']}초, 발화당 {r['model_ms_per_utterance']}ms / 모델 로드 포함 전체 {r['total_seconds']}초

## 학습 기록 (내부 dev Accuracy@0.5)
- 1단계 동결: {t['frozen']['accuracy']:.4f}
""" + ''.join(f"- 2단계 미세조정 epoch {h['epoch']}: {h['accuracy']:.4f} (train loss {h['train_loss']:.4f})\n" for h in t['tune_history'])
(LOGS/'efficiency_report.md').write_text(report, encoding='utf-8')
print('\n' + report)

I0000 00:00:1791643704.952875   15381 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791643707.396717   15381 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791643707.397244   15381 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[14:48:34] 발화 목록: 통화 3,640개 / 발화 111,947개
[14:50:43] 저장: /content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u12/mission2_validation.csv (111,947행, 읽을 수 없어 1로 채운 발화 0개)
[14:50:43] 추론 시간: 128.7s (발화당 1.15ms, 그중 모델 forward 0.55ms), batch size 32, device cuda, 모델 로드 포함 전체 134.4s

형식 확인 통과 | 통화 3,640개 · 발화 111

## 8. 제출 폴더 생성과 오프라인 확인

```text
mission2/
├── inference.py · model.py · train.py · prepare_data.py · requirements.txt · README.md
├── checkpoints/mission2.ckpt      # WavLM 설정·가중치 + 분류기 (한 파일)
└── logs/                          # 학습·추론 로그, 고른 통화 목록, efficiency_report.md
```

제출 폴더를 zip으로 묶고, 임시 폴더에 풀어 **인터넷을 막은 상태**에서 `inference.py`를 실행합니다(Validation 3통화). 원래 코드로 돌린 결과와 같은지 확인합니다.
실행을 마친 이 노트북은 저장해서 제출 폴더에 직접 넣습니다.

In [ ]:
SUB = RUN/'submission'/'mission2'
if SUB.exists():
    shutil.rmtree(SUB)
(SUB/'checkpoints').mkdir(parents=True)
for name in ['inference.py', 'model.py', 'train.py', 'prepare_data.py', 'requirements.txt', 'README.md']:
    shutil.copy2(PROJECT/name, SUB/name)
shutil.copy2(CKPT, SUB/'checkpoints'/'mission2.ckpt')
shutil.copytree(LOGS, SUB/'logs')
archive = RUN/'mission2_submission.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SUB.rglob('*')):
        if path.is_file():
            z.write(path, path.relative_to(SUB.parent))
print('생성:', archive, f'({archive.stat().st_size / 1e6:.0f}MB)')

# 오프라인 확인: 압축을 푼 제출물을 다른 작업 폴더에서, 인터넷을 막고 실행
with tempfile.TemporaryDirectory() as temp:
    temp = Path(temp)
    (temp/'in/wav').mkdir(parents=True); (temp/'in/json').mkdir()
    for label in sorted((DATA/'validation/json').glob('*.json'))[:3]:
        shutil.copy2(label, temp/'in/json'/label.name)
        shutil.copy2(DATA/'validation/wav'/(label.stem + '.wav'), temp/'in/wav'/(label.stem + '.wav'))
    with zipfile.ZipFile(archive) as z:
        z.extractall(temp/'pkg')
    offline = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HTTP_PROXY': 'http://127.0.0.1:9',
               'HTTPS_PROXY': 'http://127.0.0.1:9', 'http_proxy': 'http://127.0.0.1:9', 'https_proxy': 'http://127.0.0.1:9'}
    def infer(code_dir, checkpoint, output):
        result = subprocess.run([sys.executable, str(code_dir/'inference.py'), '--audio_dir', str(temp/'in/wav'),
                                 '--label_dir', str(temp/'in/json'), '--ckpt_path', str(checkpoint), '--output', str(output)],
                                capture_output=True, text=True, cwd=temp, env=offline)
        print(result.stdout[-600:], result.stderr[-600:])
        result.check_returncode()
    infer(temp/'pkg'/'mission2', temp/'pkg'/'mission2'/'checkpoints'/'mission2.ckpt', temp/'package.csv')
    infer(PROJECT, CKPT, temp/'reference.csv')
    same = (temp/'package.csv').read_bytes() == (temp/'reference.csv').read_bytes()
    rows = len((temp/'package.csv').read_text(encoding='utf-8').splitlines()) - 1
    if not same:
        raise RuntimeError('압축을 푼 제출물의 결과가 원래 코드의 결과와 다릅니다')
    print(f'압축 해제 → 인터넷 차단 → inference.py 실행 성공 ({rows}행, 원래 코드와 같은 결과)')
print('\n실행을 마친 이 노트북을 저장해서 제출 폴더에 넣어 주세요:', SUB)

생성: /content/drive/MyDrive/DCC_Mission2/runs/mission2_official_u12/mission2_submission.zip (110MB)
[14:51:23] 발화 목록: 통화 3개 / 발화 90개
[14:51:24] 저장: /tmp/tmpbvgle053/package.csv (90행, 읽을 수 없어 1로 채운 발화 0개)
[14:51:24] 추론 시간: 1.5s (발화당 17.04ms, 그중 모델 forward 15.25ms), batch size 32, device cuda, 모델 로드 포함 전체 2.5s
 rom different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791643880.580955   16156 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791643880.581549   16156 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

[14:51:36] 발화 목록: 통화 3개 / 발화 90개
[14:51:38] 저장: /tmp/tmpbvgle053/reference.csv (90행, 읽을 수 없어 1로 채운 발화 0개)
[14:51:38] 추론 시간: 1.6s (발화당 17.87ms, 그중 모델 forward 16.11ms), batch siz